# Setup dan instalasi

In [ ]:
'''
from google.colab import drive
drive.mount('/content/gdrive')

# Ganti path ini dengan lokasi folder PDF Anda di Google Drive
# Contoh: '/content/gdrive/MyDrive/MyPDFs'
GDrive_PDF_FOLDER_PATH = '/content/gdrive/MyDrive/PDFs_Untuk_RAG'
'''

"\nfrom google.colab import drive\ndrive.mount('/content/gdrive')\n\n# Ganti path ini dengan lokasi folder PDF Anda di Google Drive\n# Contoh: '/content/gdrive/MyDrive/MyPDFs'\nGDrive_PDF_FOLDER_PATH = '/content/gdrive/MyDrive/PDFs_Untuk_RAG'\n"

In [1]:
!pip install -qU langchain langchain-community langchain-text-splitters faiss-cpu langchain-google-genai PyMuPDF langchain-groq langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.0/73.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sour

# Instalasi Embedding dan FAISS

In [2]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os
from google.colab import userdata

# Simpan API key di: Colab Secrets (🔑) → nama secret: GEMINI
GEMINI = userdata.get('GEMINI')
os.environ["GOOGLE_API_KEY"] = GEMINI

# text-embedding-004 adalah model embedding terbaru dari Google
# menghasilkan vector 768 dimensi per teks
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2", output_dimensionality=768)

# Part 2: RAG Pipeline dengan LangChain & LangGraph


In [3]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document
import fitz  # PyMuPDF

##Langkah 1: Extract dan Chunk Dokumen PDF

In [4]:

'''
# (1) Membaca semua file di folder Google Drive dan mengunduhnya ke /content/
import glob

# Hapus file PDF lama di /content/ jika ada, untuk menghindari duplikasi
for f in glob.glob('/content/*.pdf'):
    os.remove(f)

# Ganti path ini dengan lokasi folder PDF Anda di Google Drive
# Contoh: '/content/gdrive/MyDrive/MyPDFs'
GDrive_PDF_FOLDER_PATH = '/content/gdrive/MyDrive/PDFs_Untuk_RAG'

# Salin semua file PDF dari Google Drive ke direktori /content/
# Ini memastikan bahwa glob.glob berikutnya akan menemukan file Anda.
print(f"Menyalin file PDF dari {GDrive_PDF_FOLDER_PATH} ke /content/")
for pdf_file_path in glob.glob(os.path.join(GDrive_PDF_FOLDER_PATH, '*.pdf')):
    shutil.copy(pdf_file_path, '/content/')
    print(f"  - Disalin: {os.path.basename(pdf_file_path)}")

# Temukan semua file PDF di direktori '/content/'
pdf_files = glob.glob("/content/*.pdf")

# Gabungkan teks dari semua PDF
all_pdf_text = []
for pdf_file in pdf_files:
    print(f"Mengekstrak teks dari: {pdf_file}")
    all_pdf_text.append(extract_text_from_pdf(pdf_file))

pdf_text = "\n".join(all_pdf_text)
'''

# (2) Membaca file PDF di content
#pdf_text = extract_text_from_pdf("Panduan Ringkas Registrasi Mahasiswa PPG Kemdikdasmen BGT 1 tahun 2026.pdf")


# (3) Mengunduh 1 file dari link Gdrive
import gdown
import os

# Ganti dengan Google Drive File ID Anda
# Contoh: https://drive.google.com/file/d/11cucT4zgOyvmkjkikCUWSCg1DiPTRb1J/view
# File ID-nya adalah 11cucT4zgOyvmkjkikCUWSCg1DiPTRb1J
gdrive_file_id = '1e9ySjEBagtr3NT9amWkyZqsAs51v-uAl'
pdf_filename = 'downloaded_pdf.pdf'

try:
    # Unduh file
    gdown.download(id=gdrive_file_id, output=pdf_filename, quiet=False)
    print(f"File '{pdf_filename}' berhasil diunduh.")

    # Verifikasi apakah file ada
    if os.path.exists(pdf_filename):
        print(f"Ukuran file: {os.path.getsize(pdf_filename)} bytes")
    else:
        print("File tidak ditemukan setelah diunduh.")

except Exception as e:
    print(f"Terjadi kesalahan saat mengunduh file: {e}")
    print("Pastikan File ID benar dan file diatur sebagai 'Anyone with the link' (public).")





Downloading...
From: https://drive.google.com/uc?id=1e9ySjEBagtr3NT9amWkyZqsAs51v-uAl
To: /content/downloaded_pdf.pdf
100%|██████████| 3.00M/3.00M [00:00<00:00, 134MB/s]

File 'downloaded_pdf.pdf' berhasil diunduh.
Ukuran file: 3002308 bytes


In [5]:
# Ekstrak dokumen

def extract_text_from_pdf(pdf_path):
    """Mengekstrak teks dari PDF menggunakan PyMuPDF"""
    doc = fitz.open(pdf_path)
    text = "\n".join([page.get_text() for page in doc])
    return text

pdf_text = extract_text_from_pdf(pdf_filename)

print(f"Total karakter dari semua PDF: {len(pdf_text)}")
#print("\nPreview 500 karakter pertama:")
#print(pdf_text[:500])

Total karakter dari semua PDF: 47017


In [6]:
splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_text(pdf_text)
print(f"Total chunks: {len(chunks)}")

'''
# Preview setiap chunk (100 karakter pertama saja)
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk[:100]}...")
'''

Total chunks: 59


'\n# Preview setiap chunk (100 karakter pertama saja)\nfor i, chunk in enumerate(chunks):\n    print(f"Chunk {i+1}: {chunk[:100]}...")\n'

##Langkah 2: Upload Chunk ke Vector Store

In [7]:
# Upload chunk ke vector store

import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

# Inisialisasi FAISS baru untuk dokumen PDF ini
index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))
#index = faiss.IndexFlatL2(len(embeddings.embed_query(chunks[0])))

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [8]:
from uuid import uuid4
from langchain_core.documents import Document
import time

# Bungkus setiap chunk sebagai Document object
documents = [Document(page_content=chunk) for chunk in chunks]

# Generate UUID untuk setiap dokumen
uuids = [str(uuid4()) for _ in range(len(documents))]

# Define batch size and delay
batch_size = 50 # You can adjust this value based on your quota limits
delay_between_batches_seconds = 2 # Adjust as needed

print(f"Adding {len(documents)} chunks to vector store in batches...")

for i in range(0, len(documents), batch_size):
    batch_documents = documents[i : i + batch_size]
    batch_ids = uuids[i : i + batch_size]
    try:
        vector_store.add_documents(documents=batch_documents, ids=batch_ids)
        print(f"Successfully added batch {i//batch_size + 1}/{len(documents)//batch_size + 1} ({len(batch_documents)} documents).")
    except Exception as e:
        print(f"Error adding batch {i//batch_size + 1}: {e}")
        print("Retrying after a delay...")
        time.sleep(delay_between_batches_seconds * 2) # Longer delay on error
        try:
            vector_store.add_documents(documents=batch_documents, ids=batch_ids)
            print(f"Successfully re-added batch {i//batch_size + 1} after retry.")
        except Exception as retry_e:
            print(f"Failed to add batch {i//batch_size + 1} even after retry: {retry_e}")
            break # Stop processing if a batch fails twice

    if i + batch_size < len(documents):
        print(f"Waiting for {delay_between_batches_seconds} seconds before next batch...")
        time.sleep(delay_between_batches_seconds)

print("Finished adding all chunks. Len vector store:", len(vector_store))

# Simpan ke disk agar tidak perlu embed ulang
vector_store.save_local("faiss_index")
print("Index tersimpan di folder faiss_index/")


Adding 59 chunks to vector store in batches...
Error adding batch 1: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-2\nPlease retry in 55.619718835s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPer

## Menambahkan Sumber dari Video

In [9]:
import os

# Instalasi library yang dibutuhkan
# moviepy untuk memproses video, SpeechRecognition dan pydub untuk audio
# yt-dlp untuk mengunduh video dari YouTube atau sumber lain
!pip install -q moviepy SpeechRecognition pydub yt-dlp

# Memastikan ffmpeg terinstal untuk moviepy (seringkali sudah ada di Colab)
# Jika belum terinstal, jalankan baris di bawah:
# !apt-get install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 77.1 MB/s eta 0:00:00


In [10]:
# Langkah 1: Unduh Video

# Ganti URL ini dengan video yang ingin Anda gunakan sebagai sumber (harus public)
video_url = "https://drive.google.com/file/d/1BS8g2iCcR1dIER5-1h18mwjZTwo-ReNU/view?usp=drive_link"
video_filename = "downloaded_video.mp4"

print(f"Mengunduh video dari {video_url}...")
# Menggunakan yt-dlp untuk mengunduh video
!yt-dlp -o {video_filename} {video_url}
print("Video berhasil diunduh!")

Mengunduh video dari https://drive.google.com/file/d/1BS8g2iCcR1dIER5-1h18mwjZTwo-ReNU/view?usp=drive_link...
[GoogleDrive] Extracting URL: https://drive.google.com/file/d/1BS8g2iCcR1dIER5-1h18mwjZTwo-ReNU/view?usp=drive_link
[GoogleDrive] 1BS8g2iCcR1dIER5-1h18mwjZTwo-ReNU: Downloading video webpage
[GoogleDrive] 1BS8g2iCcR1dIER5-1h18mwjZTwo-ReNU: Requesting source file
[info] 1BS8g2iCcR1dIER5-1h18mwjZTwo-ReNU: Downloading 1 format(s): source
[download] Destination: downloaded_video.mp4
[download] 100% of  190.97MiB in 00:00:05 at 33.50MiB/s
Video berhasil diunduh!


In [12]:
from moviepy.editor import VideoFileClip

# Langkah 2: Ekstrak Audio dari Video

audio_filename = "extracted_audio.wav"

print(f"Mengekstrak audio ke {audio_filename}...")
video = VideoFileClip(video_filename)
video.audio.write_audiofile(audio_filename)
print("Audio berhasil diekstrak!")

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



Mengekstrak audio ke extracted_audio.wav...
MoviePy - Writing audio in extracted_audio.wav


MoviePy - Done.
Audio berhasil diekstrak!


In [13]:
import speech_recognition as sr
from pydub import AudioSegment
from pydub.silence import split_on_silence

# Langkah 3: Transkripsi Audio ke Teks

video_transcription = ""

print("Melakukan transkripsi audio... Ini mungkin memerlukan waktu tergantung panjang audio.")

r = sr.Recognizer()

# Menggunakan pydub untuk memecah audio menjadi bagian-bagian yang lebih kecil
# Ini membantu SpeechRecognition menangani file audio yang lebih panjang dan menghindari batas API.
audio = AudioSegment.from_wav(audio_filename)
chunks_audio = split_on_silence(audio,
    min_silence_len = 500, # Minimal durasi keheningan dalam milidetik
    silence_thresh = audio.dBFS - 14, # Ambang batas keheningan
    keep_silence = 500 # Pertahankan sedikit keheningan di awal/akhir chunk
)

for i, audio_chunk in enumerate(chunks_audio):
    chunk_filename = f"chunk{i}.wav"
    audio_chunk.export(chunk_filename, format="wav")
    with sr.AudioFile(chunk_filename) as source:
        audio_listened = r.record(source)
        try:
            # Menggunakan Google Web Speech API untuk transkripsi
            text = r.recognize_google(audio_listened, language='id-ID') # Gunakan bahasa Indonesia
            video_transcription += text + " "
        except sr.UnknownValueError:
            print(f"[Peringatan] Tidak dapat memahami audio di chunk {i}")
        except sr.RequestError as e:
            print(f"[Error] Gagal meminta hasil dari layanan Google Speech Recognition; {e}")

print("Transkripsi selesai!")

# Hapus file audio chunk sementara setelah selesai
for i in range(len(chunks_audio)):
    chunk_filename = f"chunk{i}.wav"
    if os.path.exists(chunk_filename):
        os.remove(chunk_filename)

print(f"Total karakter dari transkripsi video: {len(video_transcription)}")
# print("\nPreview 500 karakter pertama transkripsi:")
# print(video_transcription[:500] + "...")

Melakukan transkripsi audio... Ini mungkin memerlukan waktu tergantung panjang audio.
Transkripsi selesai!
Total karakter dari transkripsi video: 7354


In [14]:
from langchain_core.documents import Document
from uuid import uuid4

# Langkah 4: Chunk dan Tambahkan Teks Transkripsi ke Vector Store

if video_transcription:
    # Gunakan splitter yang sama seperti untuk PDF
    video_chunks = splitter.split_text(video_transcription)
    video_documents = [Document(page_content=chunk) for chunk in video_chunks]

    # Generate UUID unik untuk setiap dokumen video
    video_uuids = [str(uuid4()) for _ in range(len(video_documents))]

    # Embed dan simpan ke FAISS vector store yang sudah ada
    vector_store.add_documents(documents=video_documents, ids=video_uuids)
    print(f"{len(video_documents)} chunks dari transkripsi video berhasil disimpan ke vector store.")
else:
    print("Tidak ada transkripsi video untuk ditambahkan ke vector store.")

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-2\nPlease retry in 18.409793505s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimen

In [24]:


# Bungkus setiap chunk sebagai Document object
video_chunks = splitter.split_text(video_transcription)
video_documents = [Document(page_content=chunk) for chunk in video_chunks]

# Generate UUID untuk setiap dokumen
uuids = [str(uuid4()) for _ in range(len(video_documents))]

# Define batch size and delay
batch_size = 50 # You can adjust this value based on your quota limits
delay_between_batches_seconds = 2 # Adjust as needed

print(f"Adding {len(video_documents)} chunks to vector store in batches...")

for i in range(0, len(video_documents), batch_size):
    batch_documents = video_documents[i : i + batch_size]
    batch_ids = uuids[i : i + batch_size]
    try:
        vector_store.add_documents(documents=batch_documents, ids=batch_ids)
        print(f"Successfully added batch {i//batch_size + 1}/{len(documents)//batch_size + 1} ({len(batch_documents)} documents).")
    except Exception as e:
        print(f"Error adding batch {i//batch_size + 1}: {e}")
        print("Retrying after a delay...")
        time.sleep(delay_between_batches_seconds * 2) # Longer delay on error
        try:
            vector_store.add_documents(documents=batch_documents, ids=batch_ids)
            print(f"Successfully re-added batch {i//batch_size + 1} after retry.")
        except Exception as retry_e:
            print(f"Failed to add batch {i//batch_size + 1} even after retry: {retry_e}")
            break # Stop processing if a batch fails twice

    if i + batch_size < len(documents):
        print(f"Waiting for {delay_between_batches_seconds} seconds before next batch...")
        time.sleep(delay_between_batches_seconds)

print("Finished adding all chunks.")

# Simpan ke disk agar tidak perlu embed ulang
vector_store.save_local("faiss_index")
print("Index tersimpan di folder faiss_index/")


Adding 1 chunks to vector store in batches...
Error adding batch 1: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-2\nPlease retry in 4.264947034s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerDa

##Langkah 3: Bangun RAG dengan LangGraph

In [17]:
from langchain_core.documents import Document
from typing import List, TypedDict
from langchain_groq import ChatGroq

# State adalah "tas" yang dibawa sepanjang perjalanan graph
# Setiap node bisa membaca dan mengisi field di State
class State(TypedDict):
    question: str           # pertanyaan dari user
    context: List[Document] # hasil retrieve dari FAISS
    answer: str             # jawaban final dari LLM

# RAG Prompt

In [18]:
from langchain_classic import hub

# Tarik prompt RAG standar dari LangChain Hub
# Prompt ini punya dua variabel: {context} dan {question}
prompt = hub.pull("rlm/rag-prompt")

# Lihat isi prompt-nya
example_messages = prompt.invoke(
    {"context": "(context goes here)", "question": "(question goes here)"}
).to_messages()

assert len(example_messages) == 1
print("Isi RAG prompt:")
print(example_messages[0].content)

  prompt = hub.pull("rlm/rag-prompt")



Isi RAG prompt:
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: (question goes here) 
Context: (context goes here) 
Answer:


In [19]:
# Inisialisasi LLM — pakai Llama 3.3 70B via Groq (gratis, cepat)
# Groq menyediakan inference engine yang sangat cepat untuk model open-source

from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

MODEL = 'llama-3.3-70b-versatile'

llm = ChatGroq(
    temperature=0.8,  # 0 = deterministik, cocok untuk Q&A faktual
    model=MODEL     # bisa diganti model lain seperti gemini atau GPT
)

In [20]:
# Node 1: Retrieve — ambil dokumen relevan dari FAISS
def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"])
    return {"context": retrieved_docs}


# Node 2: Generate — buat jawaban berdasarkan context + question
def generate(state: State):
    # Gabungkan semua chunk yang diambil jadi satu string konteks
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    # Isi template prompt dengan konteks dan pertanyaan
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    # Kirim ke LLM
    response = llm.invoke(messages)
    return {"answer": response.content}

In [21]:
from langgraph.graph import START, StateGraph

# Definisikan graph: retrieve dulu, lalu generate
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

print("Graph berhasil dikompilasi!")

  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer



Graph berhasil dikompilasi!


# Jalankan RAG

In [22]:
result = graph.invoke({"question": "Dimana mendapatkan nomor pendaftaran?"})

print("=== Context yang diambil dari FAISS ===")
for i, doc in enumerate(result["context"]):
    print(f"\nChunk {i+1}:")
    print(doc.page_content[:200] + "...")

print("\n=== Jawaban LLM ===")
print(result["answer"])

=== Context yang diambil dari FAISS ===

Chunk 1:
pertama kali calon mahasiswa harus mendapatkan nomor pendaftaran Bukalah link bit.um.ac.id garis miring lapor diri strip bgt -225 masukkan SIM PKB id lalu tekan e tunggu sesaat pada baris nomor pendaf...

=== Jawaban LLM ===
Nomor pendaftaran dapat diperoleh melalui link bit.um.ac.id, kemudian masukkan SIM PKB ID dan tekan enter, tunggu sesaat, lalu catat atau salin nomor pendaftaran yang ditampilkan. 
Nomor pendaftaran ini diperlukan untuk membuat akun registrasi.
Pastikan untuk mencatat nomor pendaftaran dengan benar.


In [ ]:
stop disini

## Menjadikan Aplikasi Interaktif dengan Streamlit

## Step 1 — Install Streamlit dan pyngrok

In [25]:
# Install Streamlit dan pyngrok
!pip install -q streamlit pyngrok google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 53.9 MB/s eta 0:00:00


## Step 2 — Daftarkan ngrok Auth Token

In [26]:
from pyngrok import ngrok
from google.colab import userdata

# Ambil token dari Colab Secrets
ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
print("ngrok token berhasil dikonfigurasi!")

ngrok token berhasil dikonfigurasi!


In [27]:
## Fungsi Helper: Jalankan Streamlit + Buka Tunnel

import subprocess
import time

def run_streamlit(filename, port=8501):
    # Kill SEMUA proses streamlit, bukan hanya yang kita spawn
    subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)

    # Force-free port kalau masih ada yang nempel
    subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True)

    # Tutup semua tunnel ngrok
    ngrok.kill()

    # Tunggu port benar-benar bebas
    time.sleep(3)

    proc = subprocess.Popen(
        [
            "streamlit", "run", filename,
            "--server.headless=true",
            "--server.port", str(port),
            "--server.enableCORS=false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    time.sleep(3)

    public_url = ngrok.connect(port)
    print(f"Streamlit berjalan: {public_url}")

    return proc

# Part 2: Chatbot dengan Gemini

In [28]:
gdrive_file_id = '13eBggSFs5w8QGX0azZw7k6WJgM_by64u'
img_filename = 'lambang-UM.png'

try:
    # Unduh file
    gdown.download(id=gdrive_file_id, output=img_filename, quiet=False)
    print(f"File '{img_filename}' berhasil diunduh.")

    # Verifikasi apakah file ada
    if os.path.exists(img_filename):
        print(f"Ukuran file: {os.path.getsize(img_filename)} bytes")
    else:
        print("File tidak ditemukan setelah diunduh.")

except Exception as e:
    print(f"Terjadi kesalahan saat mengunduh file: {e}")
    print("Pastikan File ID benar dan file diatur sebagai 'Anyone with the link' (public).")

img_filename = "/content/" + img_filename


Downloading...
From: https://drive.google.com/uc?id=13eBggSFs5w8QGX0azZw7k6WJgM_by64u
To: /content/lambang-UM.png
100%|██████████| 89.5k/89.5k [00:00<00:00, 26.0MB/s]

File 'lambang-UM.png' berhasil diunduh.
Ukuran file: 89479 bytes


In [29]:
%%writefile streamlit_chat_app.py
# Import library yang dibutuhkan
import streamlit as st          # framework web app
from google import genai         # SDK Gemini dari Google

# ── 1. Konfigurasi Halaman ───────────────────────────────────────────────────
# st.title() dan st.caption() menampilkan judul dan keterangan di bagian atas
# Menggunakan st.columns untuk menempatkan logo di sebelah kiri judul
col_logo, col_title = st.columns([1, 4]) # Rasio kolom 1:4
#img_filename = "/content/" + img_filename

with col_logo:
    st.image(
        "lambang-UM.png",
        width=100 # Sesuaikan ukuran logo
    )

with col_title:
    st.title("RegisBot")
    st.caption("Chatbot berbasis Google Gemini Flash untuk membantu proses registrasi mahasiswa PPG UM")



# ── 2. Sidebar: Pengaturan App ───────────────────────────────────────────────
# Semua widget di dalam blok 'with st.sidebar:' akan muncul di panel samping
with st.sidebar:
    st.subheader("Pengaturan")

    # Kotak input untuk API key
    # type="password" menyembunyikan teks yang diketik (muncul sebagai titik-titik)
    google_api_key = st.text_input("Google AI API Key", type="password")

    # Tombol untuk mereset percakapan
    # Parameter 'help' menampilkan tooltip saat kursor diarahkan ke tombol
    reset_button = st.button("Reset Percakapan", help="Hapus semua pesan dan mulai dari awal")

# ── 3. Validasi API Key ──────────────────────────────────────────────────────
# Kalau user belum memasukkan API key, tampilkan pesan dan hentikan eksekusi
if not google_api_key:
    st.info("Masukkan Google AI API Key di sidebar untuk mulai chat.", icon="🗝️")
    # st.stop() menghentikan eksekusi skrip di titik ini
    # Kode setelah st.stop() tidak akan dijalankan
    st.stop()

# ── 4. Inisialisasi Gemini Client ────────────────────────────────────────────
# Bagian ini hanya membuat client baru kalau:
# - Client belum pernah dibuat (pertama kali app dijalankan), ATAU
# - User mengganti API key di sidebar
#
# Kenapa perlu dicek seperti ini?
# Karena setiap interaksi user menyebabkan seluruh skrip dijalankan ulang.
# Tanpa pengecekan ini, kita akan membuat client baru setiap kali user ketik pesan
# — yang artinya konteks percakapan akan hilang terus.
#
# getattr(obj, 'attr', default) = cara aman mengakses atribut yang mungkin belum ada
if ("genai_client" not in st.session_state) or (
    getattr(st.session_state, "_last_key", None) != google_api_key
):
    try:
        # Buat client Gemini baru dengan API key dari sidebar
        st.session_state.genai_client = genai.Client(api_key=google_api_key)

        # Simpan key yang dipakai — untuk deteksi perubahan key nanti
        st.session_state._last_key = google_api_key

        # Kalau key berganti, hapus session chat lama
        # .pop() menghapus key dari dict dengan aman (tidak error kalau key tidak ada)
        st.session_state.pop("chat", None)
        st.session_state.pop("messages", None)

    except Exception as e:
        st.error(f"API Key tidak valid: {e}")
        st.stop()

# ── 5. Inisialisasi Chat Session & Riwayat Pesan ────────────────────────────
# Inisialisasi chat session Gemini kalau belum ada
if "chat" not in st.session_state:
    # Buat chat session baru dengan model gemini-2.5-flash
    # Session ini menyimpan konteks percakapan di sisi Gemini
    st.session_state.chat = st.session_state.genai_client.chats.create(
        model="gemini-3.1-flash-lite"
    )

# Inisialisasi list riwayat pesan kalau belum ada
# List ini menyimpan semua pesan untuk ditampilkan kembali saat skrip dijalankan ulang
if "messages" not in st.session_state:
    st.session_state.messages = []

# ── 6. Tombol Reset ──────────────────────────────────────────────────────────
# Kalau tombol reset diklik, hapus chat session dan riwayat pesan
if reset_button:
    st.session_state.pop("chat", None)
    st.session_state.pop("messages", None)
    # st.rerun() memaksa Streamlit me-refresh halaman dari awal
    # Ini akan menjalankan ulang seluruh skrip, dan karena session sudah dihapus,
    # chat baru akan dibuat di langkah 5
    st.rerun()

# ── 7. Tampilkan Riwayat Percakapan ─────────────────────────────────────────
# Loop ini menampilkan semua pesan yang sudah ada di session_state
# Setiap kali skrip dijalankan ulang, pesan-pesan ini ditampilkan kembali
for msg in st.session_state.messages:
    # st.chat_message() membuat bubble chat dengan role yang sesuai
    # role "user" = bubble di kanan, role "assistant" = bubble di kiri
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# ── 8. Input & Respons ───────────────────────────────────────────────────────
# st.chat_input() membuat kotak input di bagian bawah halaman
# Nilai yang diketik user tersimpan di variabel 'prompt'
# Variabel ini bernilai None kalau user belum mengirim pesan
prompt = st.chat_input("Ketik pesanmu di sini...")

# Hanya jalankan bagian ini kalau user mengirim pesan
if prompt:
    # Langkah 1: Tambah pesan user ke riwayat
    st.session_state.messages.append({"role": "user", "content": prompt})

    # Langkah 2: Tampilkan bubble pesan user
    with st.chat_message("user"):
        st.markdown(prompt)

    # Langkah 3: Kirim ke Gemini dan tampilkan respons
    try:
        # Kirim pesan ke Gemini melalui chat session yang sudah ada
        # chat.send_message() secara otomatis menyertakan riwayat percakapan sebelumnya
        response = st.session_state.chat.send_message(prompt)

        # Ambil teks dari respons
        # hasattr() memeriksa apakah objek punya atribut tertentu
        # Ini mencegah error kalau format respons tidak terduga
        if hasattr(response, "text"):
            answer = response.text
        else:
            answer = str(response)

    except Exception as e:
        # Kalau ada error (misal: rate limit, koneksi putus), tampilkan pesan error
        answer = f"Terjadi error: {e}"

    # Langkah 4: Tampilkan bubble respons assistant
    with st.chat_message("assistant"):
        st.markdown(answer)

    # Langkah 5: Simpan respons ke riwayat
    st.session_state.messages.append({"role": "assistant", "content": answer})


Writing streamlit_chat_app.py


In [30]:
## jalankan streamlit
proc = run_streamlit("streamlit_chat_app.py")

Streamlit berjalan: NgrokTunnel: "https://reuse-splendor-silent.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
## hentikan proses

# Hentikan Streamlit
try:
    proc.terminate()
    print("Streamlit dihentikan.")
except:
    print("Tidak ada proses yang berjalan.")

# Tutup semua tunnel ngrok
ngrok.kill()
print("Tunnel ngrok ditutup.")